In [ ]:
import logfire

from medici.agents.agentic.query_expander import QueryExpander
from medici.common.llm.groq import GroqClient
from medici.common.services.hybrid_search import HybridSearch
from medici.common.services.qdrant import QdrantStorageService
from medici.common.services.reranker import Reranker
from medici.common.utils.config import config
from medici.ingestion.embedding import EmbeddingService

In [ ]:
logfire.configure(service_name="Reranking")

In [ ]:
query = "What is transformer in LLM"

In [ ]:
groq_client = GroqClient(timeout_seconds=30, max_retries=2)

In [ ]:
query_expander = QueryExpander(groq_client)

In [ ]:
expanded_queries = await query_expander.expand(query)

In [ ]:
expanded_queries

In [ ]:
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
)

In [ ]:
storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=config.VECTOR_SIZE,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [ ]:
hybrid_search = HybridSearch(storage_service=storage_service, embedding_service=embedding_service)

In [ ]:
hybrid_result = await hybrid_search.search(queries=expanded_queries)

In [ ]:
hybrid_result

In [ ]:
reranker = Reranker()
rerank_result = await reranker.rerank(query=query, candidates=hybrid_result)

In [ ]:
rerank_result